### Process Addresses Data

##### 1. Ingest the data into the data lakehouse - bronze_addresses
##### 2. Perform data quality checks and transform the data as required - silver_addresses_clean
##### 3. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

### DLT support Multiple Notebook with Different Languages

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
import dlt

@dlt.table(
  name="bronze_addresses",
  comment="Raw data from the customer database",
  table_properties={"quality" : "bronze"}
)  

def create_bronze_addresses():
  return (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/Volumes/circuitbox/landing/operational_data/addresses/")
    .select("*",
            F.col("_metadata.file_path").alias("input_file_path"),
            F.current_timestamp().alias("ingest_timestamp")
            )
    
    )
  

In [0]:
@dlt.table(
    name = "silver_addresses_clean",
    comment = "Cleaned Addresses Data",
    table_properties={ "quality": "silver_cleaned"}
)
@dlt.expect_or_fail("valid_cutomer_id","customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode", "LENGTH(postcode) = 5")

def create_silver_addresses_clean():
  return (
      spark.readStream.table("LIVE.bronze_addresses")
      .select(
          "customer_id",
          "address_line_1",
          "city",
          "state",
          "postcode",
          F.col("created_date").cast("date")
      )
  )





In [0]:
dlt.create_streaming_table(
    name="silver_addresses",
    comment= "SCD Type 2 addresses data",
    table_properties = {"quality" : "silver"}
)


In [0]:
dlt.apply_changes(
    target = "silver_addresses",
    source = "silver_addresses_clean",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)